### Row Level Security

Row-level security controls which rows a user can access based on conditions such as user identity, group membership, or data attributes. For example, you might restrict sales data so that users only see rows for their assigned region.

**Example:** Suppose you have a table `sales` with a column `region`. You can define a row filter so that users in the "West" region only see rows where `region = 'West'`. In SDP, you can use the `row_filter` parameter:

In [0]:
@dp.table(
  name="regional_sales",
  row_filter="region = current_user_region()" # User can only see their regions rows
)
def regional_sales():
    return spark.readStream.table("catalog.schema.sales")

In [0]:
%sql
CREATE OR REPLACE VIEW 
  regional_sales 
AS SELECT 
  *
FROM 
  catalog.schema.sales
WHERE 
  region = CURRENT_USER_REGION();

`SET ROW FILTER` is a Unity Catalog SQL statement used to define row-level security policies directly on tables. It attaches a filter condition to a table, so only rows matching the condition are visible to users or groups. 

In [0]:
%sql
SET ROW FILTER ON 
  catalog.schema.sales
FOR
  user_group
USING 
  region = 'West'

This policy means members of `user_group` will only see rows where `region = 'West'` in the `sales` table.

### Column Level Security

Column-level security restricts access to specific columns, hiding sensitive fields from unauthorized users. For example, you might allow users to see sales amounts but not customer names or emails.

**Example:** Suppose your `sales` table has columns `customer_name`, `email`, and `amount`. You can create a view that only exposes the `amount` column:

In [0]:
@dp.materialized_view(
  name="public_sales"
)
def public_sales():
    return spark.read.table("catalog.schema.sales").select("amount")

In [0]:
%sql
CREATE OR REPLACE VIEW
  public_sales
AS SELECT
  amount
FROM
  catalog.schema.sales

### Column Masks

Column masks **control what values users see in specific columns, depending on who they are**. At query time, the mask replaces each reference to a column with the result of a masking function. This allows sensitive data, such as SSNs or emails, to be redacted or transformed based on user identity or role.

**Each column can have one mask.** The mask must be defined as a SQL UDF that returns a value of the same type as the column being masked. The SQL UDF can optionally [call Python or Scala UDFs](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/manually-apply#wrapper-example) to implement complex masking logic. Column masks can also take other columns as inputs, for example, to vary behavior based on multiple attributes.

Like row filters, column masks can be applied per table or managed centrally through ABAC policies. They operate at query time and integrate seamlessly with standard SQL, notebooks, and dashboards.

In [0]:
%sql
CREATE OR REPLACE FUNCTION streaming.security.mask_email(email STRING)
RETURNS STRING
DETERMINISTIC
RETURN CONCAT(
    SUBSTRING(email, 1, 1), -- Takes the first letter
    '***@', 
    SPLIT_PART(email, '@', 2) -- Takes everything after the @
);

| Feature      | Applies to        | Managed using         | Naming impact              | Best used for...                               |
|--------------|-------------------|-----------------------|----------------------------|-----------------------------------------------|
| Dynamic views | Views            | SQL logic             | Creates a new object name  | Sharing filtered data or spanning multiple tables |
| Row filters   | Tables           | ABAC or mapping tables | Table name unchanged       | Row-level access control tied to user or data tags |
| Column masks  | Tables/columns   | ABAC or mapping tables | Table name unchanged       | Redacting sensitive column data based on identity |

### **Unity Catalog Policies**

For advanced security, Unity Catalog supports row and column-level access policies using `GRANT` statements and attribute-based access control (ABAC).

In [0]:
%sql
-- Grant access to only the 'amount' column
GRANT SELECT (amount) ON TABLE catalog.schema.sales TO user_group;

-- Row-level filtering with dynamic views
CREATE OR REPLACE VIEW 
  regional_sales 
AS SELECT 
  *
FROM 
  catalog.schema.sales
WHERE 
  region = '${user.region}';

`current_user`

This function returns the username (email address) of the user executing the query, it's commonly used in row-level security to filter data so users only see their own records.

In [0]:
%sql
CREATE OR REPLACE VIEW my_data AS
SELECT *
FROM catalog.schema.table
WHERE owner_email = current_user();

Here, only rows where `owner_email` matches the logged-in user’s email are shown.

`is_account_member`

This function checks if a given user is a member of your Databricks account. It's useful for access control, such as restricting data to only account members or specific groups.

In [0]:
%sql
SELECT *
FROM catalog.schema.table
WHERE is_account_member('user@example.com');

This returns true if `user@example.com` is an account member.